# 4.2 Distribution fitting

Fitting one family by hand, as 4.1 did, does not scale past "does normal look right".
`DistributionFitter` fits every registered family at once and marks the winner by two
different criteria that do not always agree — the disagreement is usually more
informative than either number alone.

In [ ]:
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from goad_toolkit.config import DataConfig, FileConfig
from goad_toolkit.dataprocessor import CovidDataProcessor
from goad_toolkit.models import linear_model, mse, train_model
from goad_toolkit.distributions import DistributionRegistry
from goad_toolkit.analytics import DistributionFitter, FitResult, fit_table
from goad_toolkit.visualizer import (
    ComparePlot,
    ECDFPlot,
    PlotFits,
    FitPlotSettings,
    PlotSettings,
    QQPlot,
    ResidualPlot,
)

from wa_analyzer.data import load_showcase

rng = np.random.default_rng(7)

## A model, so there is a residual to fit

Deaths from positive tests, lagged and z-scored — the same worked example goad's
"Models and residuals" chapter walks through (`goad_get_concept("models and
residuals")` via the `goad` MCP server). `CovidDataProcessor` downloads and caches the
raw series once; `train_model` fits `deaths ≈ a · positivetests + b`.

In [ ]:
processor = CovidDataProcessor(FileConfig(), DataConfig())
data = processor.process()
print(data.shape)
data.head(3)

In [ ]:
X = data["positivetests"].to_numpy()
y = data["deaths_shifted"].to_numpy()

params = train_model(X, y, linear_model, mse, params=[0.01, 1.0], bounds=[(0, 1.0), (0, None)])
data["predicted"] = linear_model(X, params)
data["residual"] = y - data["predicted"]
print(f"fitted params (rate, offset): {params}")

In [ ]:
ComparePlot(PlotSettings(xlabel="date", ylabel="deaths", title="Deaths: actual vs. predicted")).plot(
    data=data, x="date", y1="deaths_shifted", y2="predicted",
)

In [ ]:
midpoint = str(data.index[len(data) // 2].date())
ResidualPlot(PlotSettings(figsize=(11, 4), xlabel="date", ylabel="error", title="Residual over time")).plot(
    data=data, x="date", y="residual", date=midpoint, datelabel="midpoint", interval=4,
)

No obvious drift or step — the model is not systematically wrong at any particular time.
That clears the first residual check goad's "Models and residuals" chapter walks
through (§6.3); what is left is the distribution of the error itself.

## Fit every family at once

`DistributionFitter().fit()` walks the whole registry and returns a `FitResult` or
`FailedFit` per family — no hand-written loop, no picking one family in advance.

In [ ]:
residual = data["residual"].dropna().to_numpy()

registry = DistributionRegistry()
fitter = DistributionFitter(registry)
results = fitter.fit(residual, discrete=False)

table = fit_table(results)
table

In [ ]:
PlotFits(PlotSettings(figsize=(12, 4), xlabel="error", ylabel="density", title="Covid residual: top 3 fits")).plot(
    data=residual, fit_results=results, fitplotsettings=FitPlotSettings(bins=30, max_fits=3),
)

## Do log-likelihood and KS agree here?

Crimson panels are the best-by-likelihood family, dark blue the best-by-KS. When they are
the same panel, both criteria agree — which is not guaranteed, and not something to assume
from the picture. Check the actual numbers:

In [ ]:
winners = table[(table["best_likelihood"]) | (table["best_ks"])]
print(winners[["distribution", "log_likelihood", "ks_pvalue", "best_likelihood", "best_ks"]])

by_ll = table.iloc[0]["distribution"]
by_ks = table.sort_values("ks_pvalue", ascending=False).iloc[0]["distribution"]
if by_ll == by_ks:
    print(f"\nBoth criteria agree on '{by_ll}' here.")
else:
    print(f"\nThey disagree: '{by_ll}' wins by log-likelihood, '{by_ks}' wins by KS.")
    print("Log-likelihood is dominated by the bulk of the data; KS is comparatively blind")
    print("in the tails. A disagreement is a hint to look at the qq-plot, not a tie-breaker")
    print("to average away.")

In [ ]:
best_ll = next(r for r in results if isinstance(r, FitResult) and r.best_likelihood)
QQPlot(PlotSettings(title=f"residual vs. fitted {best_ll.distribution}")).plot(
    data=residual, distribution=best_ll.frozen_dist,
)

## Outliers: a point in the tail is a question, not a verdict

The QQPlot above is where an outlier actually shows up first — a point that strays off
the line is the picture's way of saying "the fitted family doesn't explain this one".
What to do about it depends entirely on *why* it's there, and the standard tool for
finding it — the z-score rule — carries an assumption of its own that is easy to miss.


In [ ]:
z = (residual - residual.mean()) / residual.std()
z_flagged = np.abs(z) > 3

tail_prob = np.minimum(best_ll.frozen_dist.cdf(residual), 1 - best_ll.frozen_dist.cdf(residual))
dist_flagged = tail_prob < 0.005  # roughly the two-sided z>3 threshold, but under the fitted family

print(f"z-score (assumes normal): {z_flagged.sum()} points flagged")
print(f"tail probability under the fitted {best_ll.distribution}: {dist_flagged.sum()} points flagged")
print(f"the two rules disagree on {np.sum(z_flagged != dist_flagged)} point(s)")


**The z-score rule assumes the data is normal before it has checked.** Table above
already said the winning family here is *not* `norm` — so a threshold built for a bell
curve is being applied to a shape that isn't one, and the two rules disagree exactly
because of that. Whichever way they disagree, the fitted family's own tail probability is
the more defensible number: it is the same distribution the rest of this notebook already
committed to, not a second, unstated assumption smuggled in through a rule of thumb.


### What a flagged point can mean

Four different things can produce a flagged point, and only some of them justify
removing it:

1. **A measurement error** — a genuine mistake in how the value was recorded. Drop it,
   and say so.
2. **A real, rare event, correctly recorded.** The distribution has a tail for a reason;
   this is the tail doing its job. Keep it.
3. **A regime the model doesn't know about** — this notebook's own residual is an example:
   the days right around a reporting backlog or a public holiday behave differently, not
   because anything was measured wrong, but because the model has no term for "holiday".
4. **The wrong distribution, not a wrong point.** If *every* fit in the family disagrees
   with a handful of points in the same direction, the shape is wrong, not those points —
   which is the same lesson the pareto exercise below makes with an entire dataset instead
   of one point.

The test that actually distinguishes these: **can you name the mechanism**, the way §4.2's
reflection asks for one? "Reporting catches up after a holiday" is a mechanism. "It looked
big" is not, and is indistinguishable from case 2.


### Missing data is a different problem wearing the same clothes

A gap in a series is not an outlier — it is the *absence* of a point, and treating it as
one by silently interpolating or dropping it makes an assumption worth stating out loud.
Mean- or forward-filling assumes the missing values would have looked like their
neighbours; that is true for a sensor that briefly disconnected, and false for a chat
export where a week is missing because nobody was using the app that week. Before
imputing anything, `.isna().sum()` and a look at *when* the gaps fall is step one — a
missing week clustered around a single date is a different problem than one row missing
at random, and the fix is not the same.


## The extensibility exercise: a family goad does not ship

`pareto` is deliberately not in the default registry (goad's "Distributions" chapter, §5.2,
explains why) — lesson 4 registers it as the concrete case for "the shipped set is not the
ceiling". Author *message counts* in the IRC corpus is the natural candidate: a "rich get
richer" variable if there ever was one.

In [ ]:
irc = load_showcase("ubuntu_irc")
authors = Counter()
pattern = re.compile(r"^\[\d{2}:\d{2}\] <([^>]+)>", flags=re.MULTILINE)
for text in irc["text"]:
    for m in pattern.finditer(text):
        authors[m.group(1)] += 1

author_counts = pd.Series(authors).sort_values(ascending=False).to_numpy().astype(float)
print(f"{len(author_counts)} authors, busiest sent {author_counts.max():.0f} messages, median {np.median(author_counts):.0f}")

In [ ]:
before = fit_table(DistributionFitter(DistributionRegistry()).fit(author_counts, discrete=False))
before[["distribution", "log_likelihood", "ks_pvalue", "best_likelihood", "best_ks"]]

In [ ]:
extended = DistributionRegistry()
extended.register_distribution("pareto", stats.pareto, is_discrete=False, num_params=3)

extended_results = DistributionFitter(extended).fit(author_counts, discrete=False)
after = fit_table(extended_results)
after[["distribution", "log_likelihood", "ks_pvalue", "best_likelihood", "best_ks"]]

Registering `pareto` does not dethrone the likelihood winner — the bulk of authors sending
a handful of messages is still fit better by a gamma shape overall. But it does take the KS
crown: none of these fits are good in an absolute sense (every KS p-value here is
essentially zero — reject-everything territory typical of a dataset this size), but
relative to each other, `pareto` tracks the empirical CDF more closely than anything that
was in the registry before it existed to compete. That is the honest version of "the
shipped set was missing something": not a clean win, a *different and more informative*
disagreement between the two criteria than existed before.

In [ ]:
pareto_fit = next(r for r in extended_results if r.distribution == "pareto")
synthetic = pareto_fit.frozen_dist.rvs(len(author_counts), random_state=rng)  # ty: ignore[unresolved-attribute]

ECDFPlot(PlotSettings(title="Author message counts: empirical vs. a pareto sample of the same size")).plot(
    data=author_counts, compare=synthetic, label="empirical", compare_label="pareto sample",
)

The rank-frequency version of the same claim — the visual discriminator goad's
"Distributions" chapter (§5.7) recommends when a histogram cannot settle it:

In [ ]:
rank = np.arange(1, len(author_counts) + 1)
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.loglog(rank, author_counts, ".", color="steelblue", alpha=0.6)
ax.set_xlabel("author rank (log)")
ax.set_ylabel("messages sent (log)")
ax.set_title("Author activity, by rank")

Straight-ish over several orders of magnitude — the visual signature the fit numbers above
were quietly disagreeing about.

## Further, not assessed

Two related notebooks in [`goad_exercises`](https://github.com/raoulg/goad_exercises),
for anyone who wants to keep pulling this thread — neither is part of this course:

- **A beta distribution as a distribution over a probability** (`05_abtesting.ipynb`) is a
  genuinely good ten-minute read once distribution fitting makes sense: instead of fitting
  a shape to data, you fit a shape to *how confident you are*, and update it as evidence
  arrives. The multi-armed-bandit machinery built on top of it is a different course and
  would take a week to do properly — read the beta-distribution part, skip the rest unless
  you're curious.
- **The Metropolis-Hastings algorithm** (`02_metropolis_hastings.ipynb`) is excellent
  material for sampling from a distribution you can't write down a closed form for, and
  squarely out of scope here — DAV never builds the prior/posterior vocabulary it assumes.
  Worth it if you're heading toward Bayesian methods next.


## Reflection

1. In your own words: why does `DistributionFitter` mark a *likelihood* winner and a *KS*
   winner separately, instead of one combined score?
2. The pareto exercise above did not produce a clean "pareto wins" result. Is that a failure
   of the exercise, or exactly the point? Defend your answer with the actual numbers, not
   the vibe of the plot.
3. Pick one distribution from the registry that is not `pareto` and name a real variable —
   from any lesson so far — you would register it for, if it were also missing.